<a href="https://colab.research.google.com/github/BarisEmreRuzgar/Yapay-Zeka-Destekli-Otonom-Hasar-ve-Insan-Tespit-Sistemi/blob/main/Yapay-Zeka-Destekli-Otonom-Hasar-ve-Insan-Tespit-Sistemi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
from ultralytics import YOLO

# 1. Önceden eğitilmiş YOLOv8-nano (en hafif ve hızlı) modelini indiriyoruz
model = YOLO('yolov8n.pt')

# 2. Sistemi test etmek için 3 epoch'luk (döngü) mini bir eğitim başlatıyoruz
# (coco8.yaml, ultralytics'in içindeki otomatik inen minik bir test veri setidir)
results = model.train(data='coco8.yaml', epochs=3, imgsz=640)

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="QbvG5NRoxWhyO6Tny0Lo")
project = rf.workspace("akash-gnbdg").project("disaster-dnaqz")
version = project.version(2)
dataset = version.download("yolov8")


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="QbvG5NRoxWhyO6Tny0Lo")
project = rf.workspace("damaged-building-detection").project("damaged-buildings-vrcqx")
version = project.version(8)
dataset = version.download("yolov8")


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="QbvG5NRoxWhyO6Tny0Lo")
project = rf.workspace("damaged-building-detection").project("damaged-building-detection-zenbx")
version = project.version(2)
dataset = version.download("yolov8")


In [ ]:
import os
import shutil

dataset3 = "Damaged-Building-Detection-2"
merged_dir = "T3_Merged_Dataset"

# 1. Adım: Hedef klasörlerin var olduğundan emin ol (Yoksa otomatik oluşturur)
for split in ['train', 'valid', 'test']:
    os.makedirs(f"{merged_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{merged_dir}/{split}/labels", exist_ok=True)

# 2. Adım: 3. Veri setindeki görselleri ve etiketleri ana havuza kopyala
for split in ['train', 'valid', 'test']:
    if os.path.exists(f"{dataset3}/{split}"):
        # Görüntüleri kopyala
        for file in os.listdir(f"{dataset3}/{split}/images"):
            shutil.copy(f"{dataset3}/{split}/images/{file}", f"{merged_dir}/{split}/images/bina2_{file}")

        # Etiketleri oku, +1 kaydır ve yeni havuza yaz
        for file in os.listdir(f"{dataset3}/{split}/labels"):
            with open(f"{dataset3}/{split}/labels/{file}", 'r') as f:
                lines = f.readlines()

            with open(f"{merged_dir}/{split}/labels/bina2_{file}", 'w') as f:
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) > 0:
                        parts[0] = str(int(parts[0]) + 1)
                        f.write(" ".join(parts) + "\n")

print("3. Veri seti başarıyla ana havuza eklendi! Dev veri havuzu eğitime hazır.")

In [ ]:
import yaml

merged_dir = "T3_Merged_Dataset"

# YOLOv8 için 5 sınıflı (Multi-Class) veri haritasını oluşturuyoruz
yaml_content = {
    'path': f"/content/{merged_dir}",
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {
        0: 'person',
        1: 'destroyed',
        2: 'major-damage',
        3: 'minor-damage',
        4: 'no-damage'
    }
}

# Haritayı data.yaml olarak T3_Merged_Dataset klasörünün içine kaydediyoruz
with open(f"{merged_dir}/data.yaml", 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print("data.yaml dosyası başarıyla oluşturuldu! Eğitime geçebilirsin.")

In [ ]:
from ultralytics import YOLO

# Model mimarisini yüklüyoruz
model = YOLO('yolov8s.pt')

# Dev veri havuzuyla eğitimi ateşliyoruz
results = model.train(
    data="/content/T3_Merged_Dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,     # Erken durdurma mekanizması devrede
    project="T3_Otonom_Afet_Sistemi",
    name="iha_multiclass_v1"
)

In [ ]:
import os
import shutil
import yaml

dataset_person = "disaster-2"
dataset_bina1 = "Damaged-Buildings-8"
dataset_bina2 = "Damaged-Building-Detection-2"
merged_dir = "T3_Merged_Dataset"

# Eski birleştirilmiş klasörü temizleyip taze bir başlangıç yapıyoruz
if os.path.exists(merged_dir):
    shutil.rmtree(merged_dir)

for split in ['train', 'valid', 'test']:
    os.makedirs(f"{merged_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{merged_dir}/{split}/labels", exist_ok=True)

# 1. İNSAN VERİ SETİNİ EKLE (Sınıf ID: 0 - person)
if os.path.exists(dataset_person):
    for split in ['train', 'valid', 'test']:
        src_img = f"{dataset_person}/{split}/images"
        src_lbl = f"{dataset_person}/{split}/labels"
        if os.path.exists(src_img):
            for file in os.listdir(src_img):
                shutil.copy(os.path.join(src_img, file), f"{merged_dir}/{split}/images/person_{file}")
            for file in os.listdir(src_lbl):
                shutil.copy(os.path.join(src_lbl, file), f"{merged_dir}/{split}/labels/person_{file}")

# 2. BİNA HASARI 1. SETİ EKLE (ID'leri +1 kaydır: 1, 2, 3, 4)
if os.path.exists(dataset_bina1):
    for split in ['train', 'valid', 'test']:
        src_img = f"{dataset_bina1}/{split}/images"
        src_lbl = f"{dataset_bina1}/{split}/labels"
        if os.path.exists(src_img):
            for file in os.listdir(src_img):
                shutil.copy(os.path.join(src_img, file), f"{merged_dir}/{split}/images/bina1_{file}")
            for file in os.listdir(src_lbl):
                with open(os.path.join(src_lbl, file), 'r') as f:
                    lines = f.readlines()
                with open(f"{merged_dir}/{split}/labels/bina1_{file}", 'w') as f:
                    for line in lines:
                        parts = line.strip().split()
                        if len(parts) > 0:
                            parts[0] = str(int(parts[0]) + 1)
                            f.write(" ".join(parts) + "\n")

# 3. BİNA HASARI 2. SETİ EKLE (ID'leri +1 kaydır: 1, 2, 3, 4)
if os.path.exists(dataset_bina2):
    for split in ['train', 'valid', 'test']:
        src_img = f"{dataset_bina2}/{split}/images"
        src_lbl = f"{dataset_bina2}/{split}/labels"
        if os.path.exists(src_img):
            for file in os.listdir(src_img):
                shutil.copy(os.path.join(src_img, file), f"{merged_dir}/{split}/images/bina2_{file}")
            for file in os.listdir(src_lbl):
                with open(os.path.join(src_lbl, file), 'r') as f:
                    lines = f.readlines()
                with open(f"{merged_dir}/{split}/labels/bina2_{file}", 'w') as f:
                    for line in lines:
                        parts = line.strip().split()
                        if len(parts) > 0:
                            parts[0] = str(int(parts[0]) + 1)
                            f.write(" ".join(parts) + "\n")

# data.yaml dosyasını yeniden oluştur
yaml_content = {
    'path': f"/content/{merged_dir}",
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {
        0: 'person',
        1: 'destroyed',
        2: 'major-damage',
        3: 'minor-damage',
        4: 'no-damage'
    }
}

with open(f"{merged_dir}/data.yaml", 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print("Tüm veri setleri insan (0) ve hasar sınıfları (1-4) dahil edilerek yeniden ve dengeli bir şekilde birleştirildi!")

In [ ]:
from ultralytics import YOLO

# Küçük ama güçlü modelimizle yola devam ediyoruz
model = YOLO('yolov8s.pt')

# Dengelenmiş veri setiyle eğitimi başlatıyoruz
results = model.train(
    data="/content/T3_Merged_Dataset/data.yaml",
    epochs=120,      # Epoch sayısını biraz artırarak modele daha fazla öğrenme alanı tanıyoruz
    imgsz=640,
    batch=16,
    patience=30,     # Erken durdurma sabrını 30'a çıkardık
    project="T3_Otonom_Afet_Sistemi",
    name="iha_multiclass_v2"
)

In [ ]:
from ultralytics import YOLO

# V2 modelini yüklüyoruz
model = YOLO('/content/runs/detect/T3_Otonom_Afet_Sistemi/iha_multiclass_v2/weights/best.pt')

# conf değerini 0.10'a düşürerek modelin daha esnek tahmin yapmasını sağlıyoruz
results = model.predict(source="YOUR PHOTO SHOULD BE HERE", save=True, imgsz=640, conf=0.10)

In [ ]:
import glob
import os
from IPython.display import Image, display

# En güncel predict klasöründeki yeni görseli bul ve göster
image_files = glob.glob('/content/runs/detect/predict*/*.png') + glob.glob('/content/runs/detect/predict*/*.jpg')
if image_files:
    latest_file = max(image_files, key=os.path.getmtime)
    display(Image(filename=latest_file))
    print("Gösterilen güncel dosya yolu:", latest_file)